# STELLAR_SIM (Release v5.4, sept. 2026)
## Computational Astrobiology & Galactic Geopolitics Laboratory

This interactive Jupyter Notebook provides the experimental laboratory and simulation suite for the monograph:
> **Arcipelago Cosmico: Perché le civiltà aliene potrebbero esistere, ma restare isolate e invisibili**  
> *(Cosmic Archipelago: Why Alien Civilizations May Exist Yet Remain Isolated and Invisible)*  
> *by Giampaolo Maschietti (2026)*  
> Software License: **GNU General Public License v3.0 (GPLv3)** | Datasets: **Creative Commons CC BY 4.0**

---
### Quick Execution Guide:
- **Cell 1:** Single Run Simulation with procedural universe generation and 3D interactive Plotly HTML maps.
- **Cell 2:** Monte Carlo Batch Campaign (500 runs) with automated generation of the 8 scientific publication reports (MNRAS / A&A style).
- **Cell 3:** Deterministic Historical Replays of the 4 iconic case studies discussed in the book (Run #844, #167, #371, #126).
- **Cells 4 & 5:** Master Full Factorial $2^5$ Design of Experiments (Tactical 1,000-yr and Deep Time 10,000-yr horizons).
- **Cell 6:** Targeted Probes Suite (Super-Earth rocket traps, abiogenesis percolation, Dark Forest phase maps, Balbi's temporal asynchrony).
- **Cell 7:** Parametric Sensitivity Study on $\beta_{\mathrm{ESI}}$ validating **Solution 76** (Colonial Leapfrogging).
- **Cell 8:** Formal Statistical Validation (Wilson score interval, 100,000-sample bootstrap, and practical equivalence Z-test).

---
### CELL 1 — Single Run Simulation & 3D Interactive Maps (Outputs in `outputs/single_runs/`)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import re
import datetime
import random
import math
from collections import Counter

from utils.config_loader import load_json_config
from utils.logger import ASCIIExplorationLogger, Tee
from core.universe import UniverseGenerator
from core.civilization import determine_civilization_mobility, Planet
from core.diplomacy import DiplomaticRelation
from core.pathfinding import build_galaxy_graph, find_bfs_path, calculate_distance
from core.simulation import ExplorationSimulation
from analysis.log_parser import ExplorationLogParser
from analysis.strategist import StrategicPlanner
from visualization.plotter_3d import StarMapPlotter

# Centralized directory paths
CONFIG_DIR = "config"
OUTPUT_BASE = os.path.join("outputs", "single_runs")
LOG_DIR = os.path.join(OUTPUT_BASE, "logs")
PLAN_DIR = os.path.join(OUTPUT_BASE, "mission_plans")
LC_PLOTS_DIR = os.path.join(OUTPUT_BASE, "plots_lc")
HTML_DIR = os.path.join(OUTPUT_BASE, "html_maps")

SIM_CONFIG_PATH = os.path.join(CONFIG_DIR, "sim_config.json")
INTERACTION_CONFIG_PATH = os.path.join(CONFIG_DIR, "interaction_config.json")
NAMING_CONFIG_PATH = os.path.join(CONFIG_DIR, "naming_config.json")
PLOTLY_CONFIG_PATH = os.path.join(CONFIG_DIR, "plotly_config.json")
AI_DIPLOMACY_CONFIG_PATH = os.path.join(CONFIG_DIR, "ai_diplomacy_config.json")

def run_notebook_simulation(target_seed=None):
    print("=" * 80)
    print("     STELLAR_SIM V5.4 - MULTI-AGENT GALAXY & GEOPOLITICS SIMULATION          ")
    print("=" * 80)
    
    sim_cfg = load_json_config(SIM_CONFIG_PATH, "Sim Config", is_critical=True)
    if "earth_star_name" in sim_cfg and "sol_star_id" not in sim_cfg:
        sim_cfg["sol_star_id"] = sim_cfg["earth_star_name"]

    if target_seed is not None:
        random.seed(target_seed)
        sim_cfg["seed"] = target_seed
        print(f"[REPRODUCIBILITY] 🎯 Random generator anchored to fixed SEED: {target_seed}\n")
        
    interaction_cfg = load_json_config(INTERACTION_CONFIG_PATH, "Interaction Config", is_critical=False) or {"interaction_rules": {"earth_lc_start": 1.0}}
    naming_cfg = load_json_config(NAMING_CONFIG_PATH, "Naming Config", is_critical=False) or {}
    plotly_cfg = load_json_config(PLOTLY_CONFIG_PATH, "Plotly Config", is_critical=False) or {}
    ai_cfg = load_json_config(AI_DIPLOMACY_CONFIG_PATH, "AI Diplomacy Config", is_critical=False) or {}

    for d in [LOG_DIR, PLAN_DIR, LC_PLOTS_DIR, HTML_DIR]:
        os.makedirs(d, exist_ok=True)

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    log_filepath = os.path.join(LOG_DIR, f"exploration_console_output_{timestamp}.txt")
    
    original_stdout = sys.stdout
    log_file_handle = open(log_filepath, 'w', encoding='utf-8')
    sys.stdout = Tee(sys.stdout, log_file_handle)
    
    simulation_logger = ASCIIExplorationLogger(verbosity=1)
    sim = None 

    try:
        universe_gen = UniverseGenerator(sim_cfg, naming_cfg)
        stars_data = universe_gen.generate_star_field()
        print(f"[UNIVERSE] Star field procedurally generated: {len(stars_data)} stellar systems.")

        mobility = determine_civilization_mobility(sim_cfg)
        ship_speed = mobility["ship_speed_ly_per_year"]
        max_reach = mobility["max_reach_ly"]
        
        print(f"\n[TERRESTRIAL KINEMATIC PARAMETERS]")
        print(f" • Cruise speed: {mobility['ship_speed_c']:.3f} c ({mobility['ship_speed_ly_per_year']:.3f} ly/yr)")
        print(f" • Single hop maximum reach: {max_reach:.2f} ly\n")

        galaxy_graph = build_galaxy_graph(stars_data, max_reach)

        sim = ExplorationSimulation(
            stars_raw=stars_data,
            galaxy_graph=galaxy_graph,
            sim_params={**sim_cfg, "ship_speed_ly_per_year": ship_speed, "max_reach_ly": max_reach},
            interaction_cfg=interaction_cfg,
            naming_cfg=naming_cfg,
            logger=simulation_logger,
            ai_diplomacy_cfg=ai_cfg
        )
        
        sim.run()

    except Exception as e:
        print(f"\n[CRITICAL SIMULATION ERROR]: {e}")
        import traceback
        traceback.print_exc(file=sys.stdout)
    finally:
        sys.stdout = original_stdout
        log_file_handle.close()
        print(f"\n[LOG] Output saved to: {log_filepath}")

    if sim is not None:
        print("\n--- GENERATING 3D PLOTLY MAPS & SCIENTIFIC REPORTS ---")
        stars_dict = {
            sid: {
                'id': sid, 'position': s_obj.position, 'type': s_obj.type,
                'has_starbase': s_obj.has_starbase, 'starbase_owner': s_obj.starbase_owner,
                'starbase_status': s_obj.starbase_status, 'starbase_name': s_obj.starbase_name,
                'assigned_name': s_obj.assigned_name
            }
            for sid, s_obj in sim.stars.items()
        }
        
        sol_id = sim_cfg['sol_star_id']
        assigned_names_map = {sid: sys_obj.assigned_name for sid, sys_obj in sim.stars.items()}

        plotter = StarMapPlotter(sim_cfg, plotly_cfg)
        plotter.plot_simulation_plotly(stars_dict, galaxy_graph, sol_id, sim.agents, sim.diplomacy.trade_routes, output_dir=HTML_DIR)
        plotter.plot_trade_network_3d(stars_dict, sim.diplomacy.trade_routes, sol_id, assigned_names_map, output_dir=HTML_DIR)
        combat_counter = getattr(sim, 'cumulative_combat_sectors', Counter())
        plotter.plot_conflict_zones_3d(stars_dict, sim.diplomacy, combat_counter, sol_id, assigned_names_map, output_dir=HTML_DIR)

        logistics_reach = max(12.0, max_reach * 1.5)
        logistics_graph = build_galaxy_graph(stars_data, max_reach_ly=logistics_reach)
        initial_homeworlds = {ag.origin_system_id: ag.faction_name for ag in sim.agents if "Flotta di" not in ag.ship_name}

        for hw_id, hw_faction in initial_homeworlds.items():
            routes_details = []
            for s_id, sys_obj in sim.stars.items():
                if s_id == hw_id: continue
                if sys_obj.has_starbase and (sys_obj.starbase_owner == hw_faction or hw_faction in str(sys_obj.starbase_owner)):
                    path_ids = find_bfs_path(logistics_graph, hw_id, sys_obj.original_id) or [hw_id, sys_obj.original_id]
                    legs = [{"from_id": path_ids[i], "to_id": path_ids[i+1]} for i in range(len(path_ids)-1)]
                    routes_details.append({
                        "colony_name": f"Starbase {sys_obj.assigned_name}", "stage_name": "Starbase Orbitale",
                        "is_emancipated": False, "is_hostile": (sys_obj.starbase_status != "Leale"),
                        "is_starbase": True, "path_ids": path_ids, "detailed_legs": legs
                    })
                for p in sys_obj.planets:
                    if (p.is_colonized or p.colony_stage > 0) and (p.mother_faction == hw_faction or hw_faction in str(p.faction_name)):
                        path_ids = find_bfs_path(logistics_graph, hw_id, sys_obj.original_id) or [hw_id, sys_obj.original_id]
                        legs = [{"from_id": path_ids[i], "to_id": path_ids[i+1]} for i in range(len(path_ids)-1)]
                        is_emancipated = (p.colony_stage == Planet.STAGE_SOVEREIGN or p.is_independent)
                        rel_with_mother = sim.diplomacy.get_relation(hw_faction, p.faction_name)
                        is_hostile = is_emancipated and (p.colony_disposition in ["Ostile", "Predatoria"] or rel_with_mother == DiplomaticRelation.GUERRA)
                        routes_details.append({
                            "colony_name": p.assigned_name, "stage_name": p.colony_stage_name,
                            "is_emancipated": is_emancipated, "is_hostile": is_hostile,
                            "is_starbase": False, "path_ids": path_ids, "detailed_legs": legs
                        })

            safe_name = re.sub(r'[\W_]+', '_', hw_faction).strip('_')
            filename = os.path.join(HTML_DIR, f"rotta_colonie_{safe_name}_3d.html")
            plotter.plot_colonies_routes(stars_dict, logistics_graph, routes_details, hw_id, hw_faction, assigned_names_map, filename)

        print("\n--- GENERATING CIVILIZATION LEVEL (LC) TIMELINES (MNRAS / A&A STYLE) ---")
        plotter.plot_lc_evolution_timelines(sim, output_dir=LC_PLOTS_DIR)
        plotter.plot_astrophysical_demographics(sim, output_dir=LC_PLOTS_DIR)

    print("\nSimulation executed successfully.")

run_notebook_simulation()

---
### CELL 2 — Monte Carlo Batch Campaign (500 Runs) with Full MNRAS / A&A Plot Suite

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import copy
import time
import datetime
import multiprocessing
import pandas as pd

from utils.config_loader import load_json_config
from analysis.montecarlo import MonteCarloBatchEngine
from analysis.statistical_visualizer import StatisticalVisualizer

NUM_RUNS = 500
CPU_CORES = max(1, multiprocessing.cpu_count() - 1)
BATCH_LOG_VERBOSITY = 0  # 0 = Silent batch (CSV dataset + publication plots)

CONFIG_DIR = "config"
sim_cfg = load_json_config(os.path.join(CONFIG_DIR, "sim_config.json"), "Sim Config", is_critical=True)
interaction_cfg = load_json_config(os.path.join(CONFIG_DIR, "interaction_config.json"), "Interaction Config") or {}
naming_cfg = load_json_config(os.path.join(CONFIG_DIR, "naming_config.json"), "Naming Config") or {}
ai_cfg = load_json_config(os.path.join(CONFIG_DIR, "ai_diplomacy_config.json"), "AI Diplomacy Config") or {}

if "earth_star_name" in sim_cfg and "sol_star_id" not in sim_cfg:
    sim_cfg["sol_star_id"] = sim_cfg["earth_star_name"]

timestamp_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

engine = MonteCarloBatchEngine(
    sim_cfg=sim_cfg,
    interaction_cfg=interaction_cfg,
    naming_cfg=naming_cfg,
    ai_cfg=ai_cfg,
    base_folder=os.path.join("outputs", "batch_runs"),
    custom_timestamp=timestamp_str
)

results = engine.run_batch(
    num_iterations=NUM_RUNS, 
    processes=CPU_CORES, 
    verbosity=BATCH_LOG_VERBOSITY
)

print("\n--- GENERATING 8 STATISTICAL PUBLICATION REPORTS (MNRAS / A&A) ---")
dataset_path = os.path.join(engine.output_dir, "monte_carlo_runs_dataset.csv")

visualizer = StatisticalVisualizer(
    dataset_csv_path=dataset_path,
    output_dir=engine.output_dir
)
visualizer.generate_all_reports()

df = pd.read_csv(dataset_path)
n_contact = (df["first_contact_occurred"] == True).sum()
n_solitude = (df["first_contact_occurred"] == False).sum()
median_t = df.loc[df["first_contact_occurred"] == True, "time_to_first_contact_years"].median()
mean_colonies = df["total_earth_colonies"].mean()
mean_starbases = df["total_starbases"].mean()
mean_fleets = df["total_fleets_activated"].mean()

print("\n" + "=" * 80)
print(f" FERMI PARADOX STATISTICAL SUMMARY - STELLAR_SIM V5.4 ({NUM_RUNS} RUNS)")
print("=" * 80)
print(f" • Successful First Contact  : {n_contact:4d} / {NUM_RUNS} ({n_contact/NUM_RUNS*100:5.1f}%)")
print(f" • Cosmic Solitude / Filter   : {n_solitude:4d} / {NUM_RUNS} ({n_solitude/NUM_RUNS*100:5.1f}%)")
if pd.notna(median_t):
    print(f" • Median Contact Epoch       : {median_t:.1f} yr")
print(f" • Mean Terrestrial Colonies  : {mean_colonies:.1f} worlds/run")
print(f" • Mean Orbital Starbases     : {mean_starbases:.1f} stations/run")
print(f" • Mean Deployed Fleets       : {mean_fleets:.1f} active fleets")
print(f" • 📁 Archive Directory       : {os.path.abspath(engine.output_dir)}/")
print("=" * 80)

---
### CELL 3 — Deterministic Historical Replays of the Book's 4 Case Studies
- **Run #844 (`seed = 886`):** The Orion Diaspora (Chap. 8 - Imperial Deborah Overstretch & G/K Backbone Percolation)
- **Run #167 (`seed = 209`):** The Vespereon Conflict (Chap. 9 - Trade Alliances vs. Dark Forest Predation)
- **Run #371 (`seed = 413`):** The Cygnusvallis Relay (Chap. 10 - Solution 76 & Colonial Leapfrogging)
- **Run #126 (`seed = 168`):** The Silent Sentinels (Chap. 12 - Ghost Starbases & Biological Great Filter)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import re
import datetime
import random
import math
import pandas as pd
from collections import Counter

from utils.config_loader import load_json_config
from utils.logger import ASCIIExplorationLogger, Tee
from core.universe import UniverseGenerator
from core.civilization import determine_civilization_mobility, Planet
from core.diplomacy import DiplomaticRelation
from core.pathfinding import build_galaxy_graph, find_bfs_path, calculate_distance
from core.simulation import ExplorationSimulation
from visualization.plotter_3d import StarMapPlotter

TARGET_RUN_IDS = [844, 167, 371, 126]
VERBOSITY_LEVEL = 1

CONFIG_DIR = "config"
SIM_CONFIG_PATH = os.path.join(CONFIG_DIR, "sim_config.json")
INTERACTION_CONFIG_PATH = os.path.join(CONFIG_DIR, "interaction_config.json")
NAMING_CONFIG_PATH = os.path.join(CONFIG_DIR, "naming_config.json")
PLOTLY_CONFIG_PATH = os.path.join(CONFIG_DIR, "plotly_config.json")
AI_DIPLOMACY_CONFIG_PATH = os.path.join(CONFIG_DIR, "ai_diplomacy_config.json")

sim_cfg_template = load_json_config(SIM_CONFIG_PATH, "Sim Config", is_critical=True)
base_seed = sim_cfg_template.get("base_seed", 42)

print("=" * 80)
print(f"   STARTING DETERMINISTIC REPLAY FOR THE BOOK'S 4 CASE STUDIES: {TARGET_RUN_IDS}")
print("=" * 80 + "\n")

for idx, target_run_id in enumerate(TARGET_RUN_IDS, 1):
    target_seed = base_seed + target_run_id
    REPLAY_OUTPUT_DIR = os.path.join("outputs", "replays", f"replay_run_{target_run_id}_seed_{target_seed}")
    REPLAY_LOG_DIR = os.path.join(REPLAY_OUTPUT_DIR, "logs")
    REPLAY_PLOTS_DIR = os.path.join(REPLAY_OUTPUT_DIR, "plots")
    REPLAY_HTML_DIR = os.path.join(REPLAY_OUTPUT_DIR, "html_maps")

    for d in [REPLAY_OUTPUT_DIR, REPLAY_LOG_DIR, REPLAY_PLOTS_DIR, REPLAY_HTML_DIR]:
        os.makedirs(d, exist_ok=True)

    log_filepath = os.path.join(REPLAY_LOG_DIR, f"replay_narrative_run_{target_run_id}.txt")
    original_stdout = sys.stdout
    log_file_handle = open(log_filepath, 'w', encoding='utf-8')
    sys.stdout = Tee(sys.stdout, log_file_handle)

    sim = None
    try:
        sim_cfg = dict(sim_cfg_template)
        if "earth_star_name" in sim_cfg and "sol_star_id" not in sim_cfg:
            sim_cfg["sol_star_id"] = sim_cfg["earth_star_name"]
            
        interaction_cfg = load_json_config(INTERACTION_CONFIG_PATH, "Interaction Config", is_critical=False) or {}
        naming_cfg = load_json_config(NAMING_CONFIG_PATH, "Naming Config", is_critical=False) or {}
        plotly_cfg = load_json_config(PLOTLY_CONFIG_PATH, "Plotly Config", is_critical=False) or {}
        ai_cfg = load_json_config(AI_DIPLOMACY_CONFIG_PATH, "AI Diplomacy Config", is_critical=False) or {}

        random.seed(target_seed)
        sim_cfg["seed"] = target_seed
        simulation_logger = ASCIIExplorationLogger(verbosity=VERBOSITY_LEVEL)

        universe_gen = UniverseGenerator(sim_cfg, naming_cfg)
        stars_data = universe_gen.generate_star_field()

        mobility = determine_civilization_mobility(sim_cfg)
        ship_speed = mobility["ship_speed_ly_per_year"]
        max_reach = mobility["max_reach_ly"]
        galaxy_graph = build_galaxy_graph(stars_data, max_reach)

        sim = ExplorationSimulation(
            stars_raw=stars_data,
            galaxy_graph=galaxy_graph,
            sim_params={**sim_cfg, "ship_speed_ly_per_year": ship_speed, "max_reach_ly": max_reach},
            interaction_cfg=interaction_cfg,
            naming_cfg=naming_cfg,
            logger=simulation_logger,
            ai_diplomacy_cfg=ai_cfg
        )
        sim.run()

    except Exception as e:
        print(f"\n[REPLAY ERROR RUN #{target_run_id}]: {e}")
    finally:
        sys.stdout = original_stdout
        log_file_handle.close()

    if sim is not None:
        stars_dict = {
            sid: {
                'id': sid, 'position': s_obj.position, 'type': s_obj.type,
                'has_starbase': s_obj.has_starbase, 'starbase_owner': s_obj.starbase_owner,
                'starbase_status': s_obj.starbase_status, 'starbase_name': s_obj.starbase_name,
                'assigned_name': s_obj.assigned_name
            }
            for sid, s_obj in sim.stars.items()
        }
        sol_id = sim_cfg['sol_star_id']
        assigned_names_map = {sid: sys_obj.assigned_name for sid, sys_obj in sim.stars.items()}

        plotter = StarMapPlotter(sim_cfg, plotly_cfg)
        plotter.plot_simulation_plotly(stars_dict, galaxy_graph, sol_id, sim.agents, sim.diplomacy.trade_routes, output_dir=REPLAY_HTML_DIR)
        plotter.plot_trade_network_3d(stars_dict, sim.diplomacy.trade_routes, sol_id, assigned_names_map, output_dir=REPLAY_HTML_DIR)
        plotter.plot_lc_evolution_timelines(sim, output_dir=REPLAY_PLOTS_DIR)
        plotter.plot_astrophysical_demographics(sim, output_dir=REPLAY_PLOTS_DIR)
        print(f"[OK] Run #{target_run_id} (Seed {target_seed}) completed. Directory: {REPLAY_OUTPUT_DIR}/")

print("\n" + "=" * 80)
print("ALL CASE STUDIES SUCCESSFULLY REPRODUCED.")
print("=" * 80)

---
### CELL 4 — Full Factorial $2^5$ DOE (Tactical 1,000-yr Horizon — 34 Treatments x 50 Reps = 1,700 Runs)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from utils.config_loader import load_json_config
from analysis.doe_engine import DOEManager

CONFIG_DIR = "config"
sim_cfg = load_json_config(os.path.join(CONFIG_DIR, "sim_config.json"), "Sim Config", is_critical=True)
interaction_cfg = load_json_config(os.path.join(CONFIG_DIR, "interaction_config.json"), "Interaction Config") or {}
naming_cfg = load_json_config(os.path.join(CONFIG_DIR, "naming_config.json"), "Naming Config") or {}
ai_cfg = load_json_config(os.path.join(CONFIG_DIR, "ai_diplomacy_config.json"), "AI Diplomacy Config") or {}

if "earth_star_name" in sim_cfg and "sol_star_id" not in sim_cfg:
    sim_cfg["sol_star_id"] = sim_cfg["earth_star_name"]

RUNS_PER_TREATMENT = 50
OUTPUT_ROOT = os.path.join("outputs", "doe_campaign")

doe_engine = DOEManager(
    base_sim_cfg=sim_cfg,
    base_interaction_cfg=interaction_cfg,
    base_naming_cfg=naming_cfg,
    base_ai_cfg=ai_cfg,
    output_root=OUTPUT_ROOT
)

df_full_doe = doe_engine.execute_doe_campaign(
    runs_per_treatment=RUNS_PER_TREATMENT, 
    temporal_mode="TACTICAL_ACCELERATED"
)

print(f"\n[COMPLETED] Pareto charts saved to: {os.path.abspath(doe_engine.plots_dir)}/")

---
### CELL 5 — Master Deep Time Astrophysical DOE (10,000-yr Horizon — 34 Treatments x 50 Reps = 1,700 Runs)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from utils.config_loader import load_json_config
from analysis.doe_engine import DOEManager

CONFIG_DIR = "config"
sim_cfg = load_json_config(os.path.join(CONFIG_DIR, "sim_config.json"), "Sim Config", is_critical=True)
interaction_cfg = load_json_config(os.path.join(CONFIG_DIR, "interaction_config.json"), "Interaction Config") or {}
naming_cfg = load_json_config(os.path.join(CONFIG_DIR, "naming_config.json"), "Naming Config") or {}
ai_cfg = load_json_config(os.path.join(CONFIG_DIR, "ai_diplomacy_config.json"), "AI Diplomacy Config") or {}

if "earth_star_name" in sim_cfg and "sol_star_id" not in sim_cfg:
    sim_cfg["sol_star_id"] = sim_cfg["earth_star_name"]

RUNS_PER_TREATMENT = 50
OUTPUT_ROOT = os.path.join("outputs", "doe_deep_time")

doe_engine_deep = DOEManager(
    base_sim_cfg=sim_cfg,
    base_interaction_cfg=interaction_cfg,
    base_naming_cfg=naming_cfg,
    base_ai_cfg=ai_cfg,
    output_root=OUTPUT_ROOT
)

df_deep_doe = doe_engine_deep.execute_doe_campaign(
    runs_per_treatment=RUNS_PER_TREATMENT, 
    temporal_mode="DEEP_TIME_ASTROPHYSICAL"
)

print(f"\n[COMPLETED] Deep Time Pareto charts saved to: {os.path.abspath(doe_engine_deep.plots_dir)}/")

---
### CELL 6 — The 4 Targeted Probes (Stephen Webb Fermi Benchmark)
- **Probe 1:** Super-Earth Gravitational Rocket Trap (Solution 38)
- **Probe 2:** Biotic Percolation & Abiogenesis Threshold $f_{\mathrm{life}}$ (Solutions 55 & 64)
- **Probe 3:** 2D Phase Maps for Dark Forest Predation $\Theta_{\mathrm{DarkForest}}$ (Solutions 16 & 17)
- **Probe 4:** Spatiotemporal Asynchrony in Disjoint Light Cones (Solution 66 - Balbi & Ćirković)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import time
from analysis.probes_engine import TargetedProbesEngine

probes_engine = TargetedProbesEngine(
    config_dir="config",
    output_root=os.path.join("outputs", "probes_campaign")
)

start_time = time.time()
print("================================================================================")
print("     EXECUTING 4 TARGETED PROBES (CATEGORY A NUMERICAL BENCHMARKS)              ")
print("================================================================================")

probes_engine.execute_all_probes()

elapsed = time.time() - start_time
print(f"\n[COMPLETED] Probes suite executed in {elapsed:.2f} s.")
print(f" 📂 CSV Data saved to    : {os.path.abspath(probes_engine.data_dir)}/")
print(f" 📊 Publication plots to : {os.path.abspath(probes_engine.plots_dir)}/")

---
### CELL 7 — Parametric Sensitivity Study on $\beta_{\mathrm{ESI}}$ (Solution 76 & Technological Leapfrogging)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.config_loader import load_json_config
from analysis.montecarlo import run_single_monte_carlo_iteration

CONFIG_DIR = "config"
OUTPUT_SENSITIVITY = os.path.join("outputs", "sensitivity_studies_v54")
os.makedirs(OUTPUT_SENSITIVITY, exist_ok=True)
LOG_SENS_DIR = os.path.join(OUTPUT_SENSITIVITY, "logs")
os.makedirs(LOG_SENS_DIR, exist_ok=True)

sim_cfg_base = load_json_config(os.path.join(CONFIG_DIR, "sim_config.json"), "Sim Config", is_critical=True)
interaction_cfg = load_json_config(os.path.join(CONFIG_DIR, "interaction_config.json"), "Interaction Config") or {}
naming_cfg = load_json_config(os.path.join(CONFIG_DIR, "naming_config.json"), "Naming Config") or {}
ai_cfg_base = load_json_config(os.path.join(CONFIG_DIR, "ai_diplomacy_config.json"), "AI Diplomacy Config") or {}

if "civilization_parameters" not in ai_cfg_base:
    ai_cfg_base["civilization_parameters"] = {}

beta_grid = np.linspace(0.1, 1.0, 10)
runs_per_beta = 100
total_simulations = len(beta_grid) * runs_per_beta
beta_results = []

print("================================================================================")
print(f"   STARTING SENSITIVITY ANALYSIS ON BETA_ESI (SOLUTION 76)")
print(f"   Beta grid: {len(beta_grid)} steps from 0.1 to 1.0 | {runs_per_beta} runs/step")
print("================================================================================\n")

global_run_id = 1
completed_sims = 0

for b_val in beta_grid:
    b_val = round(b_val, 2)
    surpassed_count = 0
    total_valid = 0
    
    for r in range(runs_per_beta):
        completed_sims += 1
        progress = completed_sims / total_simulations
        bar_length = 40
        block = int(round(bar_length * progress))
        text = f"\r[PROGRESS] [{'#' * block + '-' * (bar_length - block)}] {progress*100:.1f}% (Run {completed_sims}/{total_simulations} - Beta={b_val})"
        sys.stdout.write(text)
        sys.stdout.flush()

        local_sim = copy.deepcopy(sim_cfg_base)
        local_ai = copy.deepcopy(ai_cfg_base)
        local_ai["civilization_parameters"]["beta_esi_leapfrogging"] = b_val
        
        args = (global_run_id, total_simulations, local_sim, interaction_cfg, naming_cfg, local_ai, LOG_SENS_DIR, 0)
        res = run_single_monte_carlo_iteration(args)
        global_run_id += 1
        
        if res:
            total_valid += 1
            if res.get("colony_surpassed_earth", False):
                surpassed_count += 1
                
    pct = (surpassed_count / max(1, total_valid)) * 100
    beta_results.append({
        "beta_esi": b_val, 
        "valid_runs": total_valid, 
        "surpassed_pct": round(pct, 2)
    })

df_beta = pd.DataFrame(beta_results)
beta_csv = os.path.join(OUTPUT_SENSITIVITY, "sensitivity_beta_esi_results.csv")
df_beta.to_csv(beta_csv, index=False)

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'STIXGeneral', 'serif'],
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9.5,
    'figure.facecolor': '#ffffff',
    'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#000000',
    'grid.color': '#d3d3d3',
    'grid.linestyle': ':',
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})

fig, ax = plt.subplots(figsize=(7.0, 4.5), dpi=300)
ax.plot(df_beta["beta_esi"], df_beta["surpassed_pct"], color='#b2182b', linewidth=2.0, marker='o', markersize=6, markerfacecolor='#ffffff')
mean_pct = df_beta["surpassed_pct"].mean()
ax.axhline(mean_pct, color='#08306b', linestyle='--', linewidth=1.5, label=rf'$\mathrm{{Mean\ Probability}} \approx {mean_pct:.1f}\%$')

ax.set_xlabel(r'$\mathrm{Pioneering\ Advantage\ Coefficient,}\ \beta_{\mathrm{ESI}}$')
ax.set_ylabel(r'$\mathrm{Leapfrogging\ Probability,}\ P(\mathrm{LC}_{\mathrm{col}} > \mathrm{LC}_{\mathrm{madre}})\ [\%]$')
ax.set_title(r'$\mathrm{Sensitivity\ Analysis:\ Leapfrogging\ vs\ Ecological\ Advantage}$', pad=12)
ax.set_xlim(0.05, 1.05)
ax.set_ylim(max(0, df_beta["surpassed_pct"].min() - 10), min(100, df_beta["surpassed_pct"].max() + 10))
ax.minorticks_on()
ax.grid(True, linestyle=':', alpha=0.7)
ax.legend(loc='lower right', frameon=True, edgecolor='#000000')

plot_path = os.path.join(OUTPUT_SENSITIVITY, "plot_sensitivity_beta_esi.png")
fig.savefig(plot_path)
plt.close(fig)

print(f"\n[COMPLETED] Sensitivity plot saved to: {plot_path}")

---
### CELL 8 — Formal Statistical Validation (Wilson Score Interval, Bootstrap & Practical Equivalence)

In [ ]:
import math
import numpy as np

# 1. Experimental campaign datasets (N = 1000 runs each)
n_v53 = 1000
x_v53 = 371  # 37.1%

n_v54 = 1000
x_v54 = 386  # 38.6%

p_v53 = x_v53 / n_v53
p_v54 = x_v54 / n_v54
delta_obs = p_v54 - p_v53

# 2. Analytical 95% Wilson Score Intervals
def wilson_score_interval(x, n, conf=0.95):
    z = 1.95996  # Standard normal 95% quantile
    p_hat = x / n
    denom = 1.0 + (z**2) / n
    center = (p_hat + (z**2) / (2 * n)) / denom
    margin = (z * math.sqrt((p_hat * (1.0 - p_hat) / n) + (z**2) / (4 * (n**2)))) / denom
    return (center - margin, center + margin)

ci_wilson_v53 = wilson_score_interval(x_v53, n_v53)
ci_wilson_v54 = wilson_score_interval(x_v54, n_v54)

# 3. Vectorized Bootstrap on Difference (100,000 Replications)
rng = np.random.default_rng(42)
pop_v53 = np.array([1]*x_v53 + [0]*(n_v53 - x_v53))
pop_v54 = np.array([1]*x_v54 + [0]*(n_v54 - x_v54))

B = 100_000
boot_v53 = rng.choice(pop_v53, size=(B, n_v53), replace=True).mean(axis=1)
boot_v54 = rng.choice(pop_v54, size=(B, n_v54), replace=True).mean(axis=1)
boot_diffs = boot_v54 - boot_v53
ci_diff_95 = np.quantile(boot_diffs, [0.025, 0.975])

# 4. Two-Sample Pooled Z-Test
p_pool = (x_v53 + x_v54) / (n_v53 + n_v54)
se_pool = math.sqrt(p_pool * (1.0 - p_pool) * (1.0/n_v53 + 1.0/n_v54))
z_stat = delta_obs / se_pool
p_val = 2.0 * (1.0 - 0.5 * (1.0 + math.erf(abs(z_stat) / math.sqrt(2.0))))

# 5. Practical Equivalence Margin (Delta threshold = +/- 5%)
delta_margin = 0.05
is_practically_equivalent = (ci_diff_95[0] > -delta_margin) and (ci_diff_95[1] < delta_margin)

print("=" * 80)
print(" FORMAL STATISTICAL AUDIT REPORT: LEAPFROGGING INVARIANCE TEST")
print("=" * 80)
print(f" • Baseline Model: p = {p_v53*100:.1f}% | 95% Wilson CI: [{ci_wilson_v53[0]*100:.2f}%, {ci_wilson_v53[1]*100:.2f}%]")
print(f" • Release v5.4  : p = {p_v54*100:.1f}% | 95% Wilson CI: [{ci_wilson_v54[0]*100:.2f}%, {ci_wilson_v54[1]*100:.2f}%]")
print("-" * 80)
print(f" • Observed Difference (Delta): {delta_obs*100:+.2f} percentage points")
print(f" • 95% Bootstrap CI of Delta  : [{ci_diff_95[0]*100:+.2f}%, {ci_diff_95[1]*100:+.2f}%]")
print(f" • Two-Sample Z-Test          : z = {z_stat:.3f}, p-value = {p_val:.4f}")
print("-" * 80)
if p_val > 0.05:
    print(" -> Statistical Significance: NOT SIGNIFICANT (p > 0.05).")
    print("    The observed difference of +1.5 p.p. is fully consistent with sampling noise.")
else:
    print(" -> Statistical Significance: SIGNIFICANT (p <= 0.05).")

if is_practically_equivalent:
    print(f" -> Practical Equivalence (|Delta| < {delta_margin*100:.0f}%): CONFIRMED.")
    print(f"    The full 95% confidence interval [{ci_diff_95[0]*100:+.2f}%, {ci_diff_95[1]*100:+.2f}%] falls")
    print(f"    completely within the +/- {delta_margin*100:.0f} p.p. tolerance margin.")
print("=" * 80)